This notebook stacks the yearly classification maps, extracts the grassland class and determines the continuity of the class for each given raster pixel. 

In [ ]:
import os
import xarray as xr
import rioxarray
import glob
from dask.distributed import Client, LocalCluster

import numpy as np
import rasterio

In [ ]:
# create Dask cluster and client
cluster = LocalCluster(processes=True, n_workers=8, memory_limit='32GB')
client = Client(cluster)
print(cluster.dashboard_link)

In [ ]:
# set variables
folder = "../data/grasslvnd"

In [ ]:
# create a grassland_age folder if it doesn't exist
if not os.path.exists(os.path.join(folder, "grassland_age")):
    os.makedirs(os.path.join(folder, "grassland_age"))

In [ ]:
# find all prediction folders in the specified directory
file_pattern = folder + "/preds_*"
pred_folders = sorted(glob.glob(file_pattern))

In [ ]:
# extract years from folder names and find the latest year
years = []
for folder_path in pred_folders:
    folder_name = folder_path.split("/")[-1]
    year_str = folder_name.split("_")[-1]
    try:
        year = int(year_str)
        years.append(year)
    except ValueError:
        continue

if not years:
    raise ValueError("no valid years found in folder names.")

largest_year = max(years)
print(f"latest year is: {largest_year}")

# set the baseline folder to the one with the latest year and find all tiles in that folder
baseline_folder = f"preds_{largest_year}"
baseline_tiles = glob.glob(os.path.join(folder,baseline_folder, "*.tif"))

# function to stack tiles for a given basename across all years, ensuring all years are present
def stack_tiles_for_basename(basename, years, pred_folders):
    tile_paths = []
    present_years = []
    for year, folder in zip(years, pred_folders):
        tile_path = os.path.join(folder, basename)
        if os.path.exists(tile_path):
            tile_paths.append(tile_path)
            present_years.append(year)

    if len(present_years) != len(years):
        print(f"skipping {basename}, not all years are present.")
        return None

    datasets = [rioxarray.open_rasterio(tile_path) for tile_path in tile_paths]
    stacked = xr.concat(datasets, dim="time")
    stacked["time"] = present_years
    return stacked

# stack tiles for all basenames in the baseline folder, ensuring all years are present
stacked_tiles = {}
for tile_path in baseline_tiles:
    basename = os.path.basename(tile_path)
    stacked = stack_tiles_for_basename(basename, years, pred_folders)
    if stacked is not None:
        stacked_tiles[basename] = stacked

In [ ]:
# determine grassland age for each stacked tile and save as new rasters
grassland_class = 3 
for basename, stacked_tile in stacked_tiles.items():
    is_grassland = (stacked_tile == grassland_class).any(dim="band")

    age = xr.zeros_like(stacked_tile.isel(time=0, band=0), dtype=np.int32)

    is_grassland_reversed = is_grassland.isel(time=slice(None, None, -1))

    consecutive_mask = is_grassland_reversed.cumprod(dim="time")

    age_data = consecutive_mask.sum(dim="time")

    baseline_grassland = is_grassland.isel(time=-1)
    age = age.where(baseline_grassland, 0)
    age.data = age_data.data
    age.rio.to_raster(os.path.join(folder, "grassland_age", f"age_{basename}"),
                      dtype=rasterio.uint8,
                      count=1,
                      compress='lzw')